# 🏠 Previsão de Preços de Imóveis

Notebook original reorganizado para ficar mais legível, mantendo a mesma lógica de modelagem.

## 1. Importações e Setup

In [ ]:
# Importando as bibliotecas principais para análise de dados, gráficos e modelagem.
import pandas as pd
import pathlib
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Importando os modelos e métricas que serão usados ao longo do notebook.
from sklearn.model_selection import train_test_split
from sklearn import linear_model
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import PolynomialFeatures
from sklearn import metrics
from sklearn.model_selection import cross_val_score

# Configurando avisos para não poluir a saída com mensagens de FutureWarning.
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings('ignore')

## 2. Carregando os Dados

In [ ]:
# Carregando a base de dados de imóveis que está no próprio repositório.
data = pd.read_csv('house_sales.csv')

# Visualizando as primeiras linhas para confirmar se a leitura deu certo.
data.head()

## 3. Análise Exploratória (EDA)

In [ ]:
# Contando quantos imóveis existem por número de quartos e exibindo em gráfico de barras.
data['num_bed'].value_counts().sort_index().plot(kind='bar')
plt.title('Distribuição por número de quartos')
plt.xlabel('Quantidade de quartos')
plt.ylabel('Contagem de imóveis')
plt.show()

In [ ]:
# Selecionando as colunas numéricas principais para explorar melhor a distribuição dos dados.
df = data[['price', 'num_bed', 'num_bath', 'size_house',
    'size_lot', 'num_floors', 'is_waterfront', 'condition', 'size_basement',
    'year_built', 'renovation_date', 'zip', 'latitude', 'longitude',
    'avg_size_neighbor_houses', 'avg_size_neighbor_lot']]

# Gerando histogramas para entender concentração e dispersão das variáveis.
h = df.hist(bins=25, figsize=(16, 16), xlabelsize='10', ylabelsize='10', xrot=-15)
plt.suptitle('Distribuição das variáveis do dataset', fontsize=16)
[x.title.set_size(12) for x in h.ravel()]
[x.yaxis.tick_left() for x in h.ravel()]
plt.tight_layout()
plt.show()

## 4. Pré-processamento

In [ ]:
# Definindo uma função auxiliar para calcular o R² ajustado.
def adjustedR2(r2, n, k):
    return r2 - (k - 1) / (n - k) * (1 - r2)

# Criando a tabela onde vou guardar os resultados de cada modelo testado.
evaluation = pd.DataFrame({'Model': [],
                           'Details': [],
                           'Root Mean Squared Error (RMSE)': [],
                           'R-squared (training)': [],
                           'Adjusted R-squared (training)': [],
                           'R-squared (test)': [],
                           'Adjusted R-squared (test)': [],
                           '5-Fold Cross Validation': []})

# Separando os dados em treino e teste (80/20).
train_data, test_data = train_test_split(df, train_size=0.8, random_state=3)

## 5. Modelagem

In [ ]:
# Treinando uma regressão linear simples usando apenas o tamanho da casa.
lr = linear_model.LinearRegression()
X_train = np.array(train_data['size_house'], dtype=pd.Series).reshape(-1, 1)
y_train = np.array(train_data['price'], dtype=pd.Series)
lr.fit(X_train, y_train)

# Preparando os dados de teste e fazendo previsões.
X_test = np.array(test_data['size_house'], dtype=pd.Series).reshape(-1, 1)
y_test = np.array(test_data['price'], dtype=pd.Series)
pred = lr.predict(X_test)

# Calculando métricas para comparar desempenho.
rmsesm = float(format(np.sqrt(metrics.mean_squared_error(y_test, pred)), '.3f'))
rtrsm = float(format(lr.score(X_train, y_train), '.3f'))
rtesm = float(format(lr.score(X_test, y_test), '.3f'))
cv = float(format(cross_val_score(lr, df[['size_house']], df['price'], cv=5).mean(), '.3f'))

print('Average Price for Test Data: {:.3f}'.format(y_test.mean()))
print('Intercept: {}'.format(lr.intercept_))
print('Coefficient: {}'.format(lr.coef_))

r = evaluation.shape[0]
evaluation.loc[r] = ['Simple Linear Regression', '-', rmsesm, rtrsm, '-', rtesm, '-', cv]
evaluation

In [ ]:
# Visualizando a relação entre tamanho da casa e preço com a linha de regressão.
sns.set(style='white', font_scale=1)
plt.figure(figsize=(6.5, 5))
plt.scatter(X_test, y_test, color='blue', label='Dados', alpha=.1)
plt.plot(X_test, lr.predict(X_test), color='red', label='Linha de regressão estimada')
plt.title('Regressão linear simples: tamanho da casa vs preço')
plt.xlabel('Tamanho da casa', fontsize=15)
plt.ylabel('Preço', fontsize=15)
plt.xticks(fontsize=13)
plt.yticks(fontsize=13)
plt.legend()
plt.gca().spines['right'].set_visible(False)
plt.gca().spines['top'].set_visible(False)
plt.show()

In [ ]:
# Copiando o dataframe para testar um modelo com mais variáveis (regressão múltipla).
df_dm = df.copy()
df_dm.describe()

# Separando novamente em treino e teste.
train_data_dm, test_data_dm = train_test_split(df_dm, train_size=0.8, random_state=3)

# Primeiro conjunto de variáveis selecionadas.
features = ['num_bed', 'num_bath', 'size_house', 'size_lot', 'num_floors', 'zip']
complex_model_1 = linear_model.LinearRegression()
complex_model_1.fit(train_data_dm[features], train_data_dm['price'])

print('Intercept: {}'.format(complex_model_1.intercept_))
print('Coefficients: {}'.format(complex_model_1.coef_))

pred = complex_model_1.predict(test_data_dm[features])
rmsecm = float(format(np.sqrt(metrics.mean_squared_error(test_data_dm['price'], pred)), '.3f'))
rtrcm = float(format(complex_model_1.score(train_data_dm[features], train_data_dm['price']), '.3f'))
artrcm = float(format(adjustedR2(complex_model_1.score(train_data_dm[features], train_data_dm['price']), train_data_dm.shape[0], len(features)), '.3f'))
rtecm = float(format(complex_model_1.score(test_data_dm[features], test_data_dm['price']), '.3f'))
artecm = float(format(adjustedR2(complex_model_1.score(test_data_dm[features], test_data['price']), test_data_dm.shape[0], len(features)), '.3f'))
cv = float(format(cross_val_score(complex_model_1, df_dm[features], df_dm['price'], cv=5).mean(), '.3f'))

r = evaluation.shape[0]
evaluation.loc[r] = ['Multiple Regression-1', 'selected features', rmsecm, rtrcm, artrcm, rtecm, artecm, cv]
evaluation.sort_values(by='5-Fold Cross Validation', ascending=False)

In [ ]:
# Adicionando mais variáveis para ver se o desempenho melhora.
features = ['num_bed', 'num_bath', 'size_house', 'size_lot', 'num_floors', 'zip', 'is_waterfront', 'condition', 'size_basement']
complex_model_2 = linear_model.LinearRegression()
complex_model_2.fit(train_data_dm[features], train_data_dm['price'])

print('Intercept: {}'.format(complex_model_2.intercept_))
print('Coefficients: {}'.format(complex_model_2.coef_))

pred = complex_model_2.predict(test_data_dm[features])
rmsecm = float(format(np.sqrt(metrics.mean_squared_error(test_data_dm['price'], pred)), '.3f'))
rtrcm = float(format(complex_model_2.score(train_data_dm[features], train_data_dm['price']), '.3f'))
artrcm = float(format(adjustedR2(complex_model_2.score(train_data_dm[features], train_data_dm['price']), train_data_dm.shape[0], len(features)), '.3f'))
rtecm = float(format(complex_model_2.score(test_data_dm[features], test_data_dm['price']), '.3f'))
artecm = float(format(adjustedR2(complex_model_2.score(test_data_dm[features], test_data_dm['price']), test_data_dm.shape[0], len(features)), '.3f'))
cv = float(format(cross_val_score(complex_model_2, df_dm[features], df_dm['price'], cv=5).mean(), '.3f'))

r = evaluation.shape[0]
evaluation.loc[r] = ['Multiple Regression-2', 'selected features', rmsecm, rtrcm, artrcm, rtecm, artecm, cv]
evaluation.sort_values(by='5-Fold Cross Validation', ascending=False)

In [ ]:
# Testando um modelo com praticamente todas as variáveis disponíveis, sem pré-processamento extra.
features = ['num_bed', 'num_bath', 'size_house', 'size_lot', 'num_floors', 'zip', 'is_waterfront', 'condition', 'size_basement', 'latitude', 'longitude', 'avg_size_neighbor_houses', 'avg_size_neighbor_lot']
complex_model_3 = linear_model.LinearRegression()
complex_model_3.fit(train_data[features], train_data['price'])

print('Intercept: {}'.format(complex_model_3.intercept_))
print('Coefficients: {}'.format(complex_model_3.coef_))

pred = complex_model_3.predict(test_data[features])
rmsecm = float(format(np.sqrt(metrics.mean_squared_error(test_data['price'], pred)), '.3f'))
rtrcm = float(format(complex_model_3.score(train_data[features], train_data['price']), '.3f'))
artrcm = float(format(adjustedR2(complex_model_3.score(train_data[features], train_data['price']), train_data.shape[0], len(features)), '.3f'))
rtecm = float(format(complex_model_3.score(test_data[features], test_data['price']), '.3f'))
artecm = float(format(adjustedR2(complex_model_3.score(test_data[features], test_data['price']), test_data.shape[0], len(features)), '.3f'))
cv = float(format(cross_val_score(complex_model_3, df[features], df['price'], cv=5).mean(), '.3f'))

r = evaluation.shape[0]
evaluation.loc[r] = ['Multiple Regression-3', 'all features, no preprocessing', rmsecm, rtrcm, artrcm, rtecm, artecm, cv]
evaluation.sort_values(by='5-Fold Cross Validation', ascending=False)

In [ ]:
# Testando regressão Ridge com diferentes valores de alpha.
features = ['num_bed', 'num_bath', 'size_house', 'size_lot', 'num_floors', 'zip', 'is_waterfront', 'condition', 'size_basement', 'latitude', 'longitude', 'avg_size_neighbor_houses', 'avg_size_neighbor_lot']

complex_model_R = linear_model.Ridge(alpha=1)
complex_model_R.fit(train_data_dm[features], train_data_dm['price'])
pred1 = complex_model_R.predict(test_data_dm[features])
rmsecm1 = float(format(np.sqrt(metrics.mean_squared_error(test_data_dm['price'], pred1)), '.3f'))
rtrcm1 = float(format(complex_model_R.score(train_data_dm[features], train_data_dm['price']), '.3f'))
artrcm1 = float(format(adjustedR2(complex_model_R.score(train_data_dm[features], train_data_dm['price']), train_data_dm.shape[0], len(features)), '.3f'))
rtecm1 = float(format(complex_model_R.score(test_data_dm[features], test_data_dm['price']), '.3f'))
artecm1 = float(format(adjustedR2(complex_model_R.score(test_data_dm[features], test_data_dm['price']), test_data_dm.shape[0], len(features)), '.3f'))
cv1 = float(format(cross_val_score(complex_model_R, df_dm[features], df_dm['price'], cv=5).mean(), '.3f'))

complex_model_R = linear_model.Ridge(alpha=100)
complex_model_R.fit(train_data_dm[features], train_data_dm['price'])
pred2 = complex_model_R.predict(test_data_dm[features])
rmsecm2 = float(format(np.sqrt(metrics.mean_squared_error(test_data_dm['price'], pred2)), '.3f'))
rtrcm2 = float(format(complex_model_R.score(train_data_dm[features], train_data_dm['price']), '.3f'))
artrcm2 = float(format(adjustedR2(complex_model_R.score(train_data_dm[features], train_data_dm['price']), train_data_dm.shape[0], len(features)), '.3f'))
rtecm2 = float(format(complex_model_R.score(test_data_dm[features], test_data_dm['price']), '.3f'))
artecm2 = float(format(adjustedR2(complex_model_R.score(test_data_dm[features], test_data_dm['price']), test_data_dm.shape[0], len(features)), '.3f'))
cv2 = float(format(cross_val_score(complex_model_R, df_dm[features], df_dm['price'], cv=5).mean(), '.3f'))

complex_model_R = linear_model.Ridge(alpha=1000)
complex_model_R.fit(train_data_dm[features], train_data_dm['price'])
pred3 = complex_model_R.predict(test_data_dm[features])
rmsecm3 = float(format(np.sqrt(metrics.mean_squared_error(test_data_dm['price'], pred3)), '.3f'))
rtrcm3 = float(format(complex_model_R.score(train_data_dm[features], train_data_dm['price']), '.3f'))
artrcm3 = float(format(adjustedR2(complex_model_R.score(train_data_dm[features], train_data_dm['price']), train_data_dm.shape[0], len(features)), '.3f'))
rtecm3 = float(format(complex_model_R.score(test_data_dm[features], test_data_dm['price']), '.3f'))
artecm3 = float(format(adjustedR2(complex_model_R.score(test_data_dm[features], test_data_dm['price']), test_data_dm.shape[0], len(features)), '.3f'))
cv3 = float(format(cross_val_score(complex_model_R, df_dm[features], df_dm['price'], cv=5).mean(), '.3f'))

r = evaluation.shape[0]
evaluation.loc[r] = ['Ridge Regression', 'alpha=1, all features', rmsecm1, rtrcm1, artrcm1, rtecm1, artecm1, cv1]
evaluation.loc[r + 1] = ['Ridge Regression', 'alpha=100, all features', rmsecm2, rtrcm2, artrcm2, rtecm2, artecm2, cv2]
evaluation.loc[r + 2] = ['Ridge Regression', 'alpha=1000, all features', rmsecm3, rtrcm3, artrcm3, rtecm3, artecm3, cv3]
evaluation.sort_values(by='5-Fold Cross Validation', ascending=False)

In [ ]:
# Criando uma nova tabela só para comparar os testes com regressão polinomial.
evaluation_poly = pd.DataFrame({'Model': [],
                                'Details': [],
                                'Root Mean Squared Error (RMSE)': [],
                                'R-squared (training)': [],
                                'Adjusted R-squared (training)': [],
                                'R-squared (test)': [],
                                'Adjusted R-squared (test)': [],
                                '5-Fold Cross Validation': []})

# Rodando modelos polinomiais com graus diferentes e conjuntos de variáveis diferentes.
features = ['num_bed', 'num_bath', 'size_house', 'size_lot', 'num_floors', 'zip', 'is_waterfront', 'condition', 'size_basement', 'latitude', 'longitude', 'avg_size_neighbor_houses', 'avg_size_neighbor_lot']
polyfeat = PolynomialFeatures(degree=2)
X_allpoly = polyfeat.fit_transform(df[features])
X_trainpoly = polyfeat.fit_transform(train_data[features])
X_testpoly = polyfeat.fit_transform(test_data[features])
poly = linear_model.LinearRegression().fit(X_trainpoly, train_data['price'])

pred1 = poly.predict(X_testpoly)
rmsepoly1 = float(format(np.sqrt(metrics.mean_squared_error(test_data['price'], pred1)), '.3f'))
rtrpoly1 = float(format(poly.score(X_trainpoly, train_data['price']), '.3f'))
rtepoly1 = float(format(poly.score(X_testpoly, test_data['price']), '.3f'))
cv1 = float(format(cross_val_score(linear_model.LinearRegression(), X_allpoly, df['price'], cv=5).mean(), '.3f'))

polyfeat = PolynomialFeatures(degree=3)
X_allpoly = polyfeat.fit_transform(df[features])
X_trainpoly = polyfeat.fit_transform(train_data[features])
X_testpoly = polyfeat.fit_transform(test_data[features])
poly = linear_model.LinearRegression().fit(X_trainpoly, train_data['price'])

pred2 = poly.predict(X_testpoly)
rmsepoly2 = float(format(np.sqrt(metrics.mean_squared_error(test_data['price'], pred2)), '.3f'))
rtrpoly2 = float(format(poly.score(X_trainpoly, train_data['price']), '.3f'))
rtepoly2 = float(format(poly.score(X_testpoly, test_data['price']), '.3f'))
cv2 = float(format(cross_val_score(linear_model.LinearRegression(), X_allpoly, df['price'], cv=5).mean(), '.3f'))

features = ['price', 'num_bed', 'num_bath', 'size_house',
    'size_lot', 'num_floors', 'is_waterfront', 'condition', 'size_basement',
    'year_built', 'renovation_date', 'zip', 'latitude', 'longitude',
    'avg_size_neighbor_houses', 'avg_size_neighbor_lot']
polyfeat = PolynomialFeatures(degree=2)
X_allpoly = polyfeat.fit_transform(df[features])
X_trainpoly = polyfeat.fit_transform(train_data[features])
X_testpoly = polyfeat.fit_transform(test_data[features])
poly = linear_model.LinearRegression().fit(X_trainpoly, train_data['price'])

pred3 = poly.predict(X_testpoly)
rmsepoly3 = float(format(np.sqrt(metrics.mean_squared_error(test_data['price'], pred3)), '.3f'))
rtrpoly3 = float(format(poly.score(X_trainpoly, train_data['price']), '.3f'))
rtepoly3 = float(format(poly.score(X_testpoly, test_data['price']), '.3f'))
cv3 = float(format(cross_val_score(linear_model.LinearRegression(), X_allpoly, df['price'], cv=5).mean(), '.3f'))

polyfeat = PolynomialFeatures(degree=3)
X_allpoly = polyfeat.fit_transform(df[features])
X_trainpoly = polyfeat.fit_transform(train_data[features])
X_testpoly = polyfeat.fit_transform(test_data[features])
poly = linear_model.LinearRegression().fit(X_trainpoly, train_data['price'])

pred4 = poly.predict(X_testpoly)
rmsepoly4 = float(format(np.sqrt(metrics.mean_squared_error(test_data['price'], pred4)), '.3f'))
rtrpoly4 = float(format(poly.score(X_trainpoly, train_data['price']), '.3f'))
rtepoly4 = float(format(poly.score(X_testpoly, test_data['price']), '.3f'))
cv4 = float(format(cross_val_score(linear_model.LinearRegression(), X_allpoly, df['price'], cv=5).mean(), '.3f'))

r = evaluation_poly.shape[0]
evaluation_poly.loc[r] = ['Polynomial Regression', 'degree=2, selected features, no preprocessing', rmsepoly1, rtrpoly1, '-', rtepoly1, '-', cv1]
evaluation_poly.loc[r + 1] = ['Polynomial Regression', 'degree=3, selected features, no preprocessing', rmsepoly2, rtrpoly2, '-', rtepoly2, '-', cv2]
evaluation_poly.loc[r + 2] = ['Polynomial Regression', 'degree=2, all features, no preprocessing', rmsepoly3, rtrpoly3, '-', rtepoly3, '-', cv3]
evaluation_poly.loc[r + 3] = ['Polynomial Regression', 'degree=3, all features, no preprocessing', rmsepoly4, rtrpoly4, '-', rtepoly4, '-', cv4]

evaluation_poly_temp = evaluation_poly[['Model', 'Details', 'Root Mean Squared Error (RMSE)', 'R-squared (training)', 'R-squared (test)', '5-Fold Cross Validation']]
evaluation_poly_temp.sort_values(by='5-Fold Cross Validation', ascending=False)

## 6. Avaliação do Modelo

In [ ]:
# Mostrando o ranking final dos modelos lineares e ridge pelo score de validação cruzada.
evaluation.sort_values(by='5-Fold Cross Validation', ascending=False)

In [ ]:
# Mostrando o ranking final dos modelos polinomiais para comparar com os modelos anteriores.
evaluation_poly_temp.sort_values(by='5-Fold Cross Validation', ascending=False)